# Exercise 1 — Fetch and plot GW data

Download open strain data and the official PE sky map for **GW190521**, then make example plots.

Run from the repo root with uv:

```bash
uv run jupyter notebook notebooks/exercises/01_fetch_plot_gw.ipynb
```

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "scripts" / "fetch_data.py").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root (scripts/fetch_data.py)")


ROOT = find_repo_root()
sys.path.insert(0, str(ROOT / "scripts"))
print("Repo root:", ROOT)

## 1a. Open strain around the merger (GWOSC via GWpy)

GW190521 GPS time ≈ `1242442967.4`. We fetch a short window of Hanford and Livingston open data.

In [ ]:
from gwpy.timeseries import TimeSeries

GPS = 1242442967.4
PAD = 16  # seconds either side

h1 = TimeSeries.fetch_open_data("H1", GPS - PAD, GPS + PAD, cache=True)
l1 = TimeSeries.fetch_open_data("L1", GPS - PAD, GPS + PAD, cache=True)
print(h1)
print(l1)

In [ ]:
h1_bp = h1.bandpass(50, 250).crop(GPS - 2, GPS + 2)
l1_bp = l1.bandpass(50, 250).crop(GPS - 2, GPS + 2)

fig, axes = plt.subplots(2, 1, figsize=(8, 4.5), sharex=True)
axes[0].plot(h1_bp.times.value - GPS, h1_bp.value * 1e21, color="C0", lw=0.8)
axes[0].set_ylabel(r"H1 strain [$10^{-21}$]")
axes[0].set_title("GW190521 — band-passed open strain (50–250 Hz)")
axes[1].plot(l1_bp.times.value - GPS, l1_bp.value * 1e21, color="C1", lw=0.8)
axes[1].set_ylabel(r"L1 strain [$10^{-21}$]")
axes[1].set_xlabel(r"$t - t_c$ [s]")
for ax in axes:
    ax.axvline(0.0, color="0.4", ls="--", lw=0.8)
fig.tight_layout()
plt.show()

## 1b. Official PE sky map

Fetch the GWOSC PE sky-map FITS with the repo helper, then plot a Mollweide projection.

In [ ]:
from fetch_data import fetch_skymaps_gwosc

paths = fetch_skymaps_gwosc(["GW190521"])
skymap_path = paths[0] if paths else ROOT / "data" / "skymaps" / "GW190521_skymap.fits"
print("Sky map:", skymap_path)
assert skymap_path.exists(), "Sky map missing — check network / GWOSC"

In [ ]:
import healpy as hp
import ligo.skymap.plot  # registers the "cylon" colormap
from ligo.skymap.io import read_sky_map
from ligo.skymap.postprocess import find_greedy_credible_levels

prob, meta = read_sky_map(str(skymap_path), nest=True)
print("Sky-map metadata keys:", sorted(meta)[:12], "...")
print("nside =", hp.npix2nside(len(prob)))

credible = find_greedy_credible_levels(prob)

plt.figure(figsize=(8, 4.2))
hp.mollview(
    prob,
    nest=True,
    title="GW190521 PE sky map",
    unit="probability density",
    cmap="cylon",
)
hp.graticule(dpar=30, dmer=60)
hp.contour(credible, nest=True, levels=[0.9], colors=["white"], linewidths=1.2)
plt.show()

area90 = (credible <= 0.9).sum() * hp.nside2pixarea(hp.npix2nside(len(prob)), degrees=True)
print(f"Approx. 90% credible area: {area90:.0f} deg²")

### Try next

- Change the band-pass or window around `GPS`.
- Fetch **GW231123** sky map the same way and compare 90% areas.
- Mark the AGN / ZTF position on this map in Exercise 3.